# m-risk · Motor de Evaluacion de Impacto Regulatorio Cross-Mercado
**Sprint 1 v7.1** — Motor base + Impacto regulatorio + Output orientado a negocio

Chile (17) · USA (8) · Japon (8) · Brasil (7) = 40 normativas

**Instrucciones:** Ejecutar Celda 1 (pip install), luego Celda 2. El Excel se guarda automaticamente.

In [ ]:
# Celda 1: Instalacion de dependencias
!pip install sentence-transformers openpyxl pandas numpy scikit-learn -q

In [ ]:
import pandas as pd
import numpy as np
import hashlib
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ── CONSTANTES ────────────────────────────────────────────────────────────────
UMBRAL_EQUIV   = 0.88
UMBRAL_PARCIAL = 0.70
# Umbrales diferenciados por categoria (Dimension 3)
UMBRAL_CATEGORIA = {
    "Acceso":          0.72,   # Estricto: consecuencias criticas
    "Acceso-Destino":  0.72,   # Igual de estricto
    "Sanitario":       0.72,   # Estricto: puede bloquear embarques
    "Inocuidad":       0.70,   # Estandar
    "Etiquetado":      0.65,   # Mas permisivo: variacion valida entre mercados
    "Ambiental":       0.68,   # Intermedio
}
PRIORIDAD = {"Acceso":1,"Acceso-Destino":1,"Sanitario":2,"Inocuidad":3,"Etiquetado":4,"Ambiental":5}

# ── CORPUS ────────────────────────────────────────────────────────────────────
rows = [
  ["Chile","LGPA Ley 18.892","SUBPESCA","Marco legal general actividades pesqueras acuicolas. Concesiones condiciones ambientales sanitarias acceso areas cultivo.","Acceso","1989","Existente","Base regulatoria salmonera Chile"],
  ["Chile","DS 319/2002","SERNAPESCA","Reglamento proteccion control erradicacion enfermedades riesgo especies hidrobiologicas RESA ISA SRS.","Sanitario","2002","Existente","Regula ISA SRS enfermedades alto riesgo. Programas Sanitarios centros cultivo obligatorios."],
  ["Chile","DS 345/2005","SERNAPESCA","Reglamento plagas hidrobiologicas REPLA. Medidas control organismos afecten centros cultivo acuicola.","Sanitario","2005","Existente","Control Caligus sea lice. Complementa RESA en materia de plagas."],
  ["Chile","DS 72/2011","SERNAPESCA","Reglamento certificacion requisitos sanitarios importacion exportacion especies hidrobiologicas.","Sanitario","2011","Existente","Base para Autorizacion Sanitaria de embarque que emite SERNAPESCA para cada exportacion."],
  ["Chile","DS 49/2006","SERNAPESCA","Reglamento centros acopio faenamiento productos hidrobiologicos condiciones sanitarias proceso planta.","Inocuidad","2006","Existente","Aplica plantas procesadoras exportadoras. Complementa HACCP exigido mercados destino."],
  ["Chile","DS 40/2012 SEIA","MMA/SEA","Reglamento Sistema Evaluacion Impacto Ambiental. EIA DGA instalacion ampliacion centros acuicolas RCA.","Ambiental","2012","Existente","RCA vigente obligatoria para operar y exportar formalmente."],
  ["Chile","Ley 21.410","SERNAPESCA","Modifica LGPA exigiendo medidas evitar deposito desechos fondos marinos planes recuperacion investigacion.","Ambiental","2022","Existente","Planes recuperacion vigentes enero 2024. Vinculada auditorias ambientales internacionales."],
  ["Chile","Autorizacion Origen Legal SERNAPESCA","SERNAPESCA","Autorizacion Origen Legal acredita cumplimiento normativa pesquera acuicola nacional por parte del exportador.","Acceso","Vigente","Existente","Obligatoria todo embarque exportacion junto Autorizacion Sanitaria de embarque."],
  ["Chile","DS 977/1996 RSA Reglamento Sanitario Alimentos","MINSAL","Reglamento Sanitario de Alimentos. Establece condiciones etiquetado rotulado de alimentos para consumo humano en Chile incluye nombre producto ingredientes fecha vencimiento peso.","Etiquetado","1996","Existente","Marco general etiquetado alimentos Chile. Base para toda exportacion. Incluye idioma espanol rotulo informacion nutricional declaracion alergenos."],
  ["Chile","Ley 20.606 Etiquetado Nutricional","MINSAL","Ley sobre composicion nutricional alimentos y su publicidad. Exige rotulo octogonal alto en calorias grasas azucares sodio en productos que superen limites establecidos.","Etiquetado","2012","Existente","Salmon procesado con aditivos sodio puede requerir sello alto en sodio. Impacto en etiquetado mercado interno y referencia para exportacion."],
  ["Chile","DS 13/2015 Reglamento Ley 20.606","MINSAL","Reglamento que establece composicion nutricional alimentos y su publicidad. Define limites nutrientes criticos sellos de advertencia octagonales obligatorios.","Etiquetado","2015","Existente","Implementacion gradual desde 2016. Salmon fresco generalmente no activa sellos pero salmon procesado ahumado marinado puede activar sodio grasas."],
  ["Chile","NCh 2840 Rotulado Productos Pesca","INN/SERNAPESCA","Norma Chilena 2840 rotulado productos de la pesca y acuicultura. Establece informacion obligatoria etiqueta nombre cientifico metodo produccion zona captura.","Etiquetado","2003","Existente","Especifica para productos pesqueros. Nombre cientifico Salmo salar obligatorio. Indicar si es cultivo salmon de cultivo o wild salmon silvestre."],
  ["Chile","Resolucion 5125/2016 Manual Sanidad Pesquera","SERNAPESCA","Manual de Sanidad Pesquera SERNAPESCA. Establece requisitos sanitarios y de etiquetado para productos pesqueros y acuicolas destinados a exportacion.","Etiquetado","2016","Existente","Especifica requisitos etiquetado exportacion segun mercado destino. Coordina con certificados sanitarios. Base para autorizacion sanitaria de embarque."],
  ["Chile","DS 320/2001 RAMA Reglamento Ambiental Acuicultura","SUBPESCA/SERNAPESCA","Reglamento Ambiental para la Acuicultura RAMA. Norma proteccion ambiental centros cultivo acuicola. Condiciones aerobicas sedimentos informes ambientales INFA caracterizacion preliminar sitio CPS. Actualizado DS 45/2021.","Ambiental","2001","Existente","Aplica todo tipo acuicultura concesiones autorizaciones inscripciones. Titulares responsables equilibrio ecologico zona concedida. Base auditorias ambientales internacionales salmonicultura."],
  ["Chile","DS 290/1993 Reglamento Concesiones Acuicultura","SUBPESCA","Reglamento de Concesiones y Autorizaciones de Acuicultura. Establece procedimientos requisitos tecnicos juridicos solicitar concesiones acuicultura areas apropiadas AAA registro nacional acuicultura.","Acceso","1993","Existente","Marco legal acceso al recurso. Sin concesion vigente no se puede operar ni exportar. Actualizado DS 114/2019. Base para toda habilitacion de centro cultivo salmon."],
  ["Chile","DS 430/1991 LGPA Texto Refundido","SUBPESCA","Decreto Supremo 430 fija texto refundido coordinado sistematizado Ley General de Pesca y Acuicultura Ley 18892. Integra todas modificaciones legislativas en texto unico oficial.","Acceso","1991","Existente","Texto refundido oficial LGPA. Referencia legal obligatoria en exportacion acuicultura. Complementa Ley 18892 con todas sus modificaciones hasta 1991."],
  ["Chile","DS 129/2013 Trazabilidad Acreditacion Origen","SERNAPESCA/SUBPESCA","Reglamento para entrega informacion pesca acuicultura y acreditacion origen legal. Establece sistema trazabilidad cadena produccion comercializacion productos hidrobiologicos exportacion.","Inocuidad","2013","Existente","Exportadores deben acreditar origen legal recursos hidrobiologicos. Sistema trazabilidad electronico SERNAPESCA. Clave para certificacion exportacion y respaldo auditorias internacionales trazabilidad."],
  ["USA","21 CFR Part 123 HACCP","FDA","Regulacion HACCP Hazard Analysis Critical Control Points procesadores pescado mariscos consumo humano exportacion.","Inocuidad","2023","Existente","Obligatoria planta extranjera exporte productos pesca USA. Plan HACCP documentado registros verificacion auditoria."],
  ["USA","FSMA Food Safety Modernization Act","FDA","Ley modernizacion inocuidad alimentaria. Registro instalaciones controles preventivos FSVP verificacion proveedores extranjeros.","Inocuidad","2016","Existente","Exportador debe estar registrado ante FDA. Importador USA aplica FSVP para verificar proveedor cumple estandares equivalentes."],
  ["USA","21 CFR Part 1 Prior Notice","FDA","Notificacion previa obligatoria ante FDA todo alimento importado USA antes arribo puerto entrada via PNSI.","Acceso-Destino","Vigente","Existente","Sin prior notice aprobada la carga no puede ingresar a USA. Presentar via sistema PNSI obligatoriamente."],
  ["USA","FSVP Foreign Supplier Verification Program","FDA","Programa verificacion proveedores extranjeros. Importador USA verifica exportador cumple estandares inocuidad equivalentes.","Acceso-Destino","2017","Existente","Exportador debe proveer evidencia cumplimiento auditorias certificados HACCP plan documentado."],
  ["USA","21 CFR Part 101 Food Labeling","FDA","Requisitos etiquetado alimentos USA. Nombre producto ingredientes alergenos informacion nutricional en ingles obligatorio.","Etiquetado","2023","Existente","Salmon declara si es cultivo o silvestre pais origen COOL metodo produccion. Alergenos declaracion obligatoria."],
  ["USA","Country of Origin Labeling COOL","USDA-AMS","Etiquetado obligatorio pais origen pescados mariscos frescos congelados refrigerados vendidos retail en USA.","Etiquetado","Vigente","Existente","Salmon chileno debe indicar Product of Chile. Aplica nivel retail supermercados."],
  ["USA","Antibiotic Residue Tolerance Levels","FDA","Limites maximos residuos antibioticos productos acuicultura importados. Oxitetraciclina antimicrobianos aprobados tolerancia.","Inocuidad","Vigente","Existente","FDA retiene embarques si detecta residuos sobre tolerancia. Chile tiene restricciones propias alineadas."],
  ["USA","Import Alert 16-131","FDA","Alerta importacion detencion automatica productos acuicolas paises historial residuos drogas no aprobadas por FDA.","Acceso","Vigente","Existente","Chile no en alerta actualmente pero riesgo debe monitorearse. Activacion implica detencion todos los embarques."],
  ["Japon","Food Sanitation Act Shokuhin Eisei Ho","MHLW","Ley Sanidad Alimentaria marco regulatorio general inocuidad alimentos Japon. Aditivos permitidos residuos pesticidas contaminantes.","Inocuidad","2024","Existente","Todo producto acuicultura importado debe cumplir esta ley. 2024 actualizo lista positiva materiales contacto alimentos. Equivalente a reglamento sanitario alimentos inocuidad alimentaria marco regulatorio general higiene."],
  ["Japon","Lista Positiva Materiales Contacto 2024","MHLW","Actualizacion 2024 sustancias permitidas envases materiales contacto alimentos importados a Japon.","Inocuidad","2024","Nuevo","Envases salmon fresco congelado deben cumplir nueva lista. Requiere revision proveedores de packaging urgente. Equivalente a norma materiales contacto alimentos envases embalajes sustancias permitidas autorizadas."],
  ["Japon","Certificado Sanitario MHLW","MHLW","Certificado sanitario emitido por autoridad competente pais exportador SERNAPESCA Chile. Requerido ingreso productos acuicultura Japon.","Sanitario","Vigente","Existente","Japon exige presentar en aduana. Sin certificado valido SERNAPESCA carga retenida para inspeccion o rechazada. Equivalente certificado sanitario oficial exportacion habilitacion sanitaria producto pesquero."],
  ["Japon","Inspeccion Cuarentena Puerto Entrada","MHLW/Aduana","Productos alimenticios importados sujetos inspeccion sanitaria al arribo. Permiso aduanero solo tras aprobar controles sanitarios.","Acceso-Destino","Vigente","Existente","Si importador japones no tiene resultados vigilancia previos Japon puede exigir importar muestras para confirmar."],
  ["Japon","Importador Domiciliado Japon Requisito Legal","MHLW","Solo empresa domiciliada Japon puede actuar como importador legal. Responsable etiquetado tramites ante MHLW y Aduana.","Acceso-Destino","Vigente","Existente","Exportador chileno necesita socio importador japones confiable. El importador responde legalmente ante MHLW."],
  ["Japon","Etiquetado JAS Ley Pesos Medidas","CAA/MAFF","Requisitos etiquetado en japones nombre producto ingredientes metodo almacenaje consumo nombre direccion importador.","Etiquetado","Vigente","Existente","Todo etiquetado debe estar en japones. Importador japones responsable adecuacion etiquetado antes comercializacion. Equivalente rotulado etiquetado alimentos nombre producto ingredientes informacion nutricional idioma local."],
  ["Japon","LMR Pesticidas Medicamentos Veterinarios Positive List","MHLW","Sistema Positive List residuos pesticidas medicamentos veterinarios. Sustancias no listadas tienen LMR de 0.01 ppm automatico.","Inocuidad","Vigente","Existente","Extremadamente estricto. Medicamentos usados en Chile deben tener LMR establecido en Japon de lo contrario 0.01 ppm. Equivalente limite maximo residuos LMR pesticidas medicamentos veterinarios antibioticos acuicultura control residuos."],
  ["Japon","Restricciones Sustancias Quimicas Clase I 2024","MHLW","Directrices actualizadas 2024 prohiben importacion productos que contengan sustancias quimicas especificadas de clase I.","Inocuidad","2024","Nuevo","Verificar insumos produccion alimento peces tratamientos no contienen sustancias de lista actualizada 2024 Japon. Equivalente restriccion sustancias quimicas prohibidas contaminantes productos acuicolas control."],
  ["Brasil","IN MAPA 1 2017 Habilitacion Planta","MAPA/DIPOA","Instruccion Normativa habilitacion sanitaria establecimientos extranjeros exportadores productos origen animal a Brasil.","Acceso","2017","Existente","Planta procesadora chilena debe estar habilitada ante DIPOA antes de tramitar Licencia Importacion. Sin habilitacion no exporta."],
  ["Brasil","Decreto 9013 2017 RIISPOA","MAPA/DIPOA","Reglamento Inspeccion Industrial Sanitaria Productos Origen Animal. Estandares calidad proceso inspeccion pescados derivados.","Inocuidad","2020","Existente","Define condiciones higienico sanitarias procesamiento. Complementado Manual Fiscalizacion Pescado Derivados MAPA."],
  ["Brasil","Licencia Importacion LI DIPOA","MAPA/DIPOA","Importacion productos origen animal requiere licencia no automatica DIPOA aprobada antes del embarque via Siscomex.","Acceso-Destino","Vigente","Existente","DIPOA realiza analisis documental puede exigir inspeccion en puerto. Retraso en aprobacion genera costos almacenaje."],
  ["Brasil","RDC ANVISA 727 2022 Alergenos","ANVISA","Rotulado obligatorio alergenos alimentarios en Brasil. Pescado es alergeno de declaracion obligatoria en etiqueta.","Etiquetado","2022","Existente","Etiquetado debe indicar presencia pescado y derivados. Aplica salmon todas sus presentaciones fresco congelado. Equivalente rotulado alergenos etiquetado obligatorio declaracion pescado gluten soya leche ingredientes."],
  ["Brasil","IN MAPA 22 2005 Etiquetado Pesca","MAPA","Norma etiquetado comercial productos pesca Brasil. Nombre producto forma presentacion conservacion establecimiento exportador.","Etiquetado","2021","Existente","Etiquetado debe estar en portugues. Formato rotulado debe ser aprobado por DIPOA antes de primera exportacion."],
  ["Brasil","PNCR Programa Control Residuos","MAPA","Programa monitoreo residuos medicamentos veterinarios contaminantes productos origen animal incluyendo pescado importado Brasil.","Inocuidad","Vigente","Existente","Brasil realiza muestreos en puerto. Residuos fuera de limite el lote es retenido destruido o reembarcado. Equivalente programa control monitoreo residuos medicamentos veterinarios antibioticos contaminantes productos pesqueros acuicultura."],
  ["Brasil","Portaria MAPA 884 2023 MoluBiS Trazabilidad","MAPA","Programa Nacional Moluscos Bivalves Seguros MoluBiS establece precedente trazabilidad extensible productos pesqueros acuicolas.","Inocuidad","2023","Nuevo","Senal regulatoria Brasil aumentando exigencias trazabilidad productos acuicolas. Monitorear para proximos ciclos. Equivalente sistema trazabilidad acreditacion origen legal cadena productiva registro informacion pesca acuicultura."],
]

cols = ["Mercado","Norma","Organismo","Requisito","Categoria","Version","Estado","Observaciones"]
df = pd.DataFrame(rows, columns=cols)
df["texto"] = df.apply(
    lambda r: f"[{r['Categoria']}] {r['Norma']} {r['Organismo']}: {r['Requisito']}. {r['Observaciones']}",
    axis=1
)
df["id"] = df.apply(
    lambda r: hashlib.md5(f"{r['Mercado']}_{r['Norma']}_{r['Version']}".encode()).hexdigest()[:10],
    axis=1
)

print(f"Corpus cargado: {len(df)} normativas")
print("Mercados:", df['Mercado'].value_counts().to_dict())
print("Estados: ", df['Estado'].value_counts().to_dict())

# ── MODELO Y EMBEDDINGS ───────────────────────────────────────────────────────
MODEL_NAME = "paraphrase-multilingual-mpnet-base-v2"
print(f"\nCargando modelo: {MODEL_NAME} (~2-3 min primera vez)...")
model = SentenceTransformer(MODEL_NAME)
print("Modelo cargado OK")

print("Generando embeddings...")
vecs = np.array(model.encode(df["texto"].tolist(), show_progress_bar=True, batch_size=16))
print(f"Embeddings generados: {vecs.shape}")

# ── FUNCIONES ─────────────────────────────────────────────────────────────────
def clasificar(score, categoria=None):
    umbral = UMBRAL_CATEGORIA.get(categoria, UMBRAL_PARCIAL) if categoria else UMBRAL_PARCIAL
    if score >= UMBRAL_EQUIV:
        return "ya_cubierta", "Bajo"
    elif score >= umbral:
        return "variacion", "Medio"
    else:
        return "gap_nuevo", "Alto"

def detectar_gaps(origen, destino):
    orig = df[df['Mercado'] == origen]
    dest = df[df['Mercado'] == destino]
    result = []
    for _, rd in dest.iterrows():
        categoria = rd['Categoria']
        # Acceso-Destino: informar pero separar de gaps del exportador
        es_destino = categoria == 'Acceso-Destino'
        umbral = UMBRAL_CATEGORIA.get(categoria, UMBRAL_PARCIAL)
        vd = vecs[rd.name].reshape(1, -1)
        # Comparar contra categoria equivalente en origen (Acceso-Destino vs Acceso)
        cat_buscar = 'Acceso' if es_destino else categoria
        oc = orig[orig['Categoria'] == cat_buscar]
        sc_max = float(cosine_similarity(vd, vecs[oc.index]).max()) if not oc.empty else 0.0
        if sc_max < umbral:
            result.append({
                "norma": rd['Norma'],
                "categoria": categoria,
                "score_max": round(sc_max, 4),
                "prioridad": PRIORIDAD.get(categoria, 9),
                "tipo": "obligacion_destino" if es_destino else "gap_exportador"
            })
    return sorted(result, key=lambda x: x['prioridad'])

def clasificar_nuevas():
    nuevas = df[df['Estado'].isin(['Nuevo', 'Modificado'])]
    base   = df[df['Estado'] == 'Existente']
    result = []
    for _, rn in nuevas.iterrows():
        vn = vecs[rn.name].reshape(1, -1)
        bc = base[(base['Mercado'] == rn['Mercado']) & (base['Categoria'] == rn['Categoria'])]
        sc_max = float(cosine_similarity(vn, vecs[bc.index]).max()) if not bc.empty else 0.0
        tipo, riesgo = clasificar(sc_max, rn['Categoria'])
        result.append({
            "norma":          rn['Norma'],
            "mercado":        rn['Mercado'],
            "categoria":      rn['Categoria'],
            "clasificacion":  tipo,
            "score":          round(sc_max, 4),
            "riesgo":         riesgo
        })
    return sorted(result, key=lambda x: PRIORIDAD.get(x['categoria'], 9))

# ── RESULTADOS ────────────────────────────────────────────────────────────────
emo = {"ya_cubierta": "OK ", "variacion": "~~", "gap_nuevo": "!!"}

print("\n" + "="*60)
print("  NORMAS NUEVAS DEL CICLO MENSUAL")
print("="*60)
for r in clasificar_nuevas():
    print(f"\n  {emo.get(r['clasificacion'],'?')} [{r['clasificacion'].upper()}]")
    print(f"     Mercado:   {r['mercado']}")
    print(f"     Norma:     {r['norma']}")
    print(f"     Categoria: {r['categoria']}  |  Score: {r['score']}  |  Riesgo: {r['riesgo']}")

for destino in ["USA", "Japon", "Brasil"]:
    gs = detectar_gaps("Chile", destino)
    print(f"\n{'='*60}")
    print(f"  GAPS Chile -> {destino}  ({len(gs)} detectados)")
    print("="*60)
    for g in gs:
        tipo_label = " [OBLIG.DESTINO]" if g.get('tipo') == 'obligacion_destino' else " [GAP EXPORTADOR]"
        print(f"  !! [{g['categoria']}]{tipo_label} {g['norma']}")
        print(f"     Score max vs Chile: {g['score_max']}")

print("\n\n  SPRINT 1 - MOTOR BASE COMPLETADO")
print(f"  Corpus: {len(df)} normativas | Embeddings: {vecs.shape}")

# ══ CAPA DE IMPACTO REGULATORIO (Sprint 4 - nueva capa) ═════════════════════

# ── Función principal de evaluación ─────────────────────────────────────────
def evaluar_impacto(score, categoria, mercado_destino=""):
    """
    Transforma score + categoria en decision de negocio.
    No modifica el pipeline existente — opera sobre sus outputs.
    """
    # PASO 1: Estado base por score (umbrales diferenciados por categoria)
    umbral_gap = UMBRAL_CATEGORIA.get(categoria, UMBRAL_PARCIAL)
    if score >= UMBRAL_EQUIV:
        estado = "Cubierto"
    elif score >= umbral_gap:
        estado = "Parcialmente cubierto"
    else:
        estado = "No cubierto"

    # PASO 2: Peso regulatorio por categoria
    if categoria in ["Acceso", "Acceso-Destino", "Sanitario"]:
        peso = "ALTO"
    elif categoria in ["Inocuidad"]:
        peso = "MEDIO"
    else:
        peso = "BAJO"

    # PASO 3: Nivel de impacto (Estado x Peso)
    matriz_impacto = {
        ("Cubierto",              "ALTO"):  ("Bajo",  "Bajo"),
        ("Cubierto",              "MEDIO"): ("Bajo",  "Bajo"),
        ("Cubierto",              "BAJO"):  ("Bajo",  "Bajo"),
        ("Parcialmente cubierto", "ALTO"):  ("Alto",  "Alto"),
        ("Parcialmente cubierto", "MEDIO"): ("Medio", "Medio"),
        ("Parcialmente cubierto", "BAJO"):  ("Bajo",  "Bajo"),
        ("No cubierto",           "ALTO"):  ("Alto",  "Alto"),
        ("No cubierto",           "MEDIO"): ("Alto",  "Alto"),
        ("No cubierto",           "BAJO"):  ("Medio", "Medio"),
    }
    nivel_impacto, riesgo = matriz_impacto[(estado, peso)]

    # PASO 4: Accion sugerida
    acciones = {
        ("No cubierto",           "ALTO"):  "Revisar si existe cobertura implicita. Si no, accion inmediata del Area Legal antes del proximo embarque.",
        ("No cubierto",           "MEDIO"): "Revisar y documentar brecha antes del proximo embarque.",
        ("No cubierto",           "BAJO"):  "Monitorear. Adecuacion recomendada en proximo ciclo.",
        ("Parcialmente cubierto", "ALTO"):  "Area Legal debe validar si el alcance cubre la obligacion. No asumir cobertura.",
        ("Parcialmente cubierto", "MEDIO"): "Revisar diferencias especificas de alcance. Documentar brecha.",
        ("Parcialmente cubierto", "BAJO"):  "Brecha menor. Confirmar con revision rapida.",
        ("Cubierto",              "ALTO"):  "Sin accion requerida. Confirmar en < 5 min.",
        ("Cubierto",              "MEDIO"): "Sin accion requerida.",
        ("Cubierto",              "BAJO"):  "Sin accion requerida.",
    }
    # Ajuste: gap Sanitario cerca del umbral = especificidad, no ausencia
    if estado == "No cubierto" and categoria == "Sanitario" and score >= 0.68:
        accion = "Gap de especificidad, no de ausencia. Verificar si el certificado existente cubre el requisito especifico del mercado destino."
    else:
        accion = acciones[(estado, peso)]

    return {
        "estado":         estado,
        "nivel_impacto":  nivel_impacto,
        "riesgo":         riesgo,
        "accion":         accion,
        "score":          round(score, 4),
        "categoria":      categoria,
        "mercado_destino": mercado_destino,
    }


# ── Evaluación de impacto sobre gaps detectados ──────────────────────────────
def evaluar_gaps_con_impacto(mercado_origen, mercado_destino):
    """
    Toma los gaps del motor existente y agrega capa de impacto regulatorio.
    Las obligaciones de destino se tratan como INFO — no son brechas del exportador.
    """
    gaps_raw = detectar_gaps(mercado_origen, mercado_destino)
    resultados = []
    for g in gaps_raw:
        if g.get("tipo") == "obligacion_destino":
            # No es brecha del exportador — impacto informativo
            impacto = {
                "estado":          "Informativo",
                "nivel_impacto":   "Info",
                "riesgo":          "Info",
                "accion":          "Obligacion del importador destino. Sin accion requerida del exportador.",
                "score":           round(g["score_max"], 4),
                "categoria":       g["categoria"],
                "mercado_destino": mercado_destino,
            }
        else:
            impacto = evaluar_impacto(
                score=g["score_max"],
                categoria=g["categoria"],
                mercado_destino=mercado_destino
            )
        resultados.append({**g, **impacto})
    return resultados


# ── Evaluación de impacto sobre normas nuevas del ciclo ──────────────────────
def evaluar_nuevas_con_impacto():
    """
    Toma las normas nuevas clasificadas y agrega capa de impacto.
    """
    nuevas_raw = clasificar_nuevas()
    resultados = []
    for n in nuevas_raw:
        impacto = evaluar_impacto(
            score=n["score"],
            categoria=n["categoria"],
            mercado_destino=n["mercado"]
        )
        resultados.append({**n, **impacto})
    return resultados


# ── Output orientado a negocio ────────────────────────────────────────────────
SEMAFORO = {"Alto": "ROJO", "Medio": "AMARILLO", "Bajo": "VERDE", "Info": "INFO"}
ICONO    = {"Cubierto": "OK", "Parcialmente cubierto": "~~", "No cubierto": "!!", "Informativo": "--"}

def print_tabla_impacto(resultados, titulo):
    """Imprime resultados como tabla orientada a negocio."""
    print("\n" + "="*70)
    print(f"  {titulo}")
    print("="*70)
    print(f"\n  {'Mercado':<10} {'Categoria':<16} {'Estado':<24} {'Impacto':<10} {'Riesgo':<10}")
    print(f"  {'-'*8:<10} {'-'*14:<16} {'-'*22:<24} {'-'*8:<10} {'-'*8:<10}")
    for r in resultados:
        mercado   = r.get("mercado", r.get("mercado_destino", ""))
        categoria = r.get("categoria", "")
        estado    = r.get("estado", "")
        impacto   = r.get("nivel_impacto", "")
        riesgo    = r.get("riesgo", "")
        accion    = r.get("accion", "")
        norma     = r.get("norma", r.get("norma_nueva", r.get("norma", "")))
        print(f"  {mercado:<10} {categoria:<16} {estado:<24} {impacto:<10} {riesgo:<10}")
        print(f"  {'':10} Norma: {norma}")
        print(f"  {'':10} Accion: {accion}")
        print()

def print_impacto_normas_nuevas():
    resultados = evaluar_nuevas_con_impacto()
    print_tabla_impacto(resultados, "IMPACTO REGULATORIO - NORMAS NUEVAS DEL CICLO")

def print_impacto_gaps(mercado_origen, mercado_destino):
    resultados = evaluar_gaps_con_impacto(mercado_origen, mercado_destino)
    altos  = [r for r in resultados if r["nivel_impacto"] == "Alto"  and r.get("tipo") == "gap_exportador"]
    medios = [r for r in resultados if r["nivel_impacto"] == "Medio" and r.get("tipo") == "gap_exportador"]
    bajos  = [r for r in resultados if r["nivel_impacto"] == "Bajo"  and r.get("tipo") == "gap_exportador"]
    info   = [r for r in resultados if r.get("tipo") == "obligacion_destino"]
    print(f"\n{'='*70}")
    print(f"  GAPS {mercado_origen} -> {mercado_destino} | ROJO: {len(altos)} | AMARILLO: {len(medios)} | VERDE: {len(bajos)} | INFO: {len(info)}")
    print("="*70)
    # Agregar campo mercado para la tabla
    for r in resultados:
        r["mercado"] = mercado_destino
    print_tabla_impacto(resultados, f"DETALLE IMPACTO {mercado_destino}")


# ── EJECUTAR CAPA DE IMPACTO ─────────────────────────────────────────────────
print_impacto_normas_nuevas()
print_impacto_gaps("Chile", "USA")
print_impacto_gaps("Chile", "Japon")
print_impacto_gaps("Chile", "Brasil")



# ── MEJORA 3: EXPORTAR TABLA DE IMPACTO A EXCEL ──────────────────────────────
print("\n" + "="*70)
print("  GENERANDO EXCEL DE IMPACTO REGULATORIO...")
print("="*70)

resultados_finales = []
for destino in ["USA", "Japon", "Brasil"]:
    gaps = evaluar_gaps_con_impacto("Chile", destino)
    for r in gaps:
        r["mercado_destino"] = destino
    resultados_finales.extend(gaps)

# Normas nuevas del ciclo
nuevas = evaluar_nuevas_con_impacto()
for n in nuevas:
    n["mercado_destino"] = n.get("mercado", "")
    n["norma"]           = n.get("norma", "")
    n["categoria"]       = n.get("categoria", "")
    n["tipo"]            = "norma_nueva"
    resultados_finales.extend(nuevas)
    break  # evitar duplicados — ya se extiende en el loop

# Construir DataFrame con columnas orientadas a negocio
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

columnas = ["Mercado", "Categoria", "Norma", "Tipo", "Estado", "Impacto", "Riesgo", "Score", "Accion", "Validado por", "Timestamp"]
filas = []
for r in resultados_finales:
    filas.append({
        "Mercado":      r.get("mercado_destino", r.get("mercado", "")),
        "Categoria":    r.get("categoria", ""),
        "Norma":        r.get("norma", ""),
        "Tipo":         "Obligacion Destino" if r.get("tipo") == "obligacion_destino" else ("Norma Nueva" if r.get("tipo") == "norma_nueva" else "Gap Exportador"),
        "Estado":       r.get("estado", ""),
        "Impacto":      r.get("nivel_impacto", ""),
        "Riesgo":       r.get("riesgo", ""),
        "Score":        r.get("score", r.get("score_max", "")),
        "Accion":       r.get("accion", ""),
        "Validado por": "",
        "Timestamp":    "",
    })

# Crear Excel con formato
wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Impacto Regulatorio"

# Header
header_fill = PatternFill("solid", fgColor="005FA6")
header_font = Font(name="Arial", bold=True, color="FFFFFF", size=11)
brd = Border(
    left=Side(style="thin", color="D1D5DB"),
    right=Side(style="thin", color="D1D5DB"),
    top=Side(style="thin", color="D1D5DB"),
    bottom=Side(style="thin", color="D1D5DB")
)

for col, nombre in enumerate(columnas, 1):
    c = ws.cell(row=1, column=col, value=nombre)
    c.font = header_font
    c.fill = header_fill
    c.alignment = Alignment(horizontal="center", vertical="center")
    c.border = brd

ws.row_dimensions[1].height = 24

# Colores por impacto
fill_rojo    = PatternFill("solid", fgColor="FEE2E2")
fill_amarillo= PatternFill("solid", fgColor="FEF3C7")
fill_verde   = PatternFill("solid", fgColor="DCFCE7")
fill_info    = PatternFill("solid", fgColor="DBEAFE")
fill_alt     = PatternFill("solid", fgColor="F8F9FA")

for row_idx, fila in enumerate(filas, 2):
    impacto = fila.get("Impacto", "")
    fill = fill_rojo if impacto == "Alto" else fill_amarillo if impacto == "Medio" else fill_verde if impacto == "Bajo" else fill_info

    for col, key in enumerate(columnas, 1):
        c = ws.cell(row=row_idx, column=col, value=fila.get(key, ""))
        c.font = Font(name="Arial", size=10)
        c.alignment = Alignment(vertical="top", wrap_text=True)
        c.border = brd
        if key in ["Estado", "Impacto", "Riesgo"]:
            c.fill = fill

    ws.row_dimensions[row_idx].height = 40

# Anchos de columna
anchos = [12, 16, 42, 18, 24, 10, 10, 8, 60, 16, 18]
for col, ancho in enumerate(anchos, 1):
    ws.column_dimensions[ws.cell(row=1, column=col).column_letter].width = ancho

ws.freeze_panes = "A2"
ws.auto_filter.ref = f"A1:{ws.cell(row=1, column=len(columnas)).column_letter}1"

nombre_archivo = "impacto_regulatorio_cross_mercado.xlsx"
wb.save(nombre_archivo)
print(f"  Excel guardado: {nombre_archivo}")
print(f"  Total filas:    {len(filas)}")
print(f"  Columnas:       {', '.join(columnas)}")

print("\n\n  MOTOR v7 COMPLETADO - Capa de impacto regulatorio activa")
print(f"  Corpus: {len(df)} normativas | Embeddings: {vecs.shape}")
